# 07 — Propensity model (T5)


In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import log_loss, accuracy_score

sys.path.insert(0, str(Path("..").resolve()))
from src.dataset import OpenBanditDataset
from src.propensity import (
    OBD_COMMON_CONTEXT_DIM,
    train_propensity_model,
    extract_propensity_scores,
    propensity_calibration_curve,
    effective_sample_size,
    overlap_diagnostics,
)

sns.set_theme(style="whitegrid")
np.random.seed(42)

N_ACTIONS = 80
N_FEATURES = OBD_COMMON_CONTEXT_DIM
RANDOM_STATE = 42
PI_EVAL = 1.0 / N_ACTIONS

FIGURES_DIR = Path("../figures/week5")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Wczytanie danych RANDOM policy


In [ ]:
dataset_random = OpenBanditDataset(behavior_policy="random", campaign="all")
feedback_random = dataset_random.obtain_batch_bandit_feedback()

context = feedback_random["context"][:, :N_FEATURES]
action = feedback_random["action"]
reward = feedback_random["reward"].astype(np.float32)

print(f"Context shape:  {context.shape}  (dim_context={dataset_random.dim_context})")
print(f"Action shape:   {action.shape}  | unique actions: {np.unique(action).size}")
print(f"Reward mean:    {reward.mean():.4f} (CTR)")
print(f"Action min/max: {action.min()} / {action.max()}")


## Trening modelu propensity scores


In [ ]:
pscore_model = train_propensity_model(
    context=feedback_random["context"],
    action=action,
    n_actions=N_ACTIONS,
    random_state=RANDOM_STATE,
    n_features=N_FEATURES,
)
print(f"Model trained. best_iteration: {pscore_model.best_iteration}")
print(f"Classes: {pscore_model.classes_.shape[0]} (expected {N_ACTIONS})")


## Diagnostyki modelu


In [ ]:
pscores_train = extract_propensity_scores(
    pscore_model, feedback_random["context"], action, n_features=N_FEATURES
)

proba_all = pscore_model.predict_proba(context)
pred_all = pscore_model.classes_[np.argmax(proba_all, axis=1)]
logloss = log_loss(action, proba_all, labels=np.arange(N_ACTIONS))
acc = accuracy_score(action, pred_all)

print(f"Multiclass log-loss: {logloss:.4f}")
print(f"Top-1 accuracy:      {acc:.4f}")
print(f"Propensity score stats (train):")
print(f"  min={pscores_train.min():.6f}  max={pscores_train.max():.6f}  mean={pscores_train.mean():.4f}")
print(f"  % NaN:         {np.isnan(pscores_train).mean()*100:.2f}%")
print(f"  % below 0.001: {(pscores_train < 0.001).mean()*100:.2f}%")


In [ ]:
fraction_pos, mean_pred = propensity_calibration_curve(
    pscore_model, feedback_random["context"], action, n_features=N_FEATURES
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.plot(mean_pred, fraction_pos, "o-", color="steelblue", label="Top-1 hit rate")
ax.set_xlabel("Mean predicted confidence (max P(a|s))")
ax.set_ylabel("Fraction correct (top-1 = observed action)")
ax.set_title("Reliability diagram — Propensity Model (top-1)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "calibration_curve.png", dpi=160)
plt.show()


In [ ]:
weights_raw = PI_EVAL / np.clip(pscores_train, 1e-9, None)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(weights_raw, bins=60, log=True, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(weights_raw.mean(), color="red", linestyle="--", label=f"Mean={weights_raw.mean():.2f}")
ax.axvline(np.percentile(weights_raw, 99), color="orange", linestyle="--", label=f"99th={np.percentile(weights_raw, 99):.2f}")
ax.set_xlabel("IPS weight π_e / P(a|s)")
ax.set_ylabel("Count (log scale)")
ax.set_title("IPS Weight Distribution — unclipped (train split)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ips_weights_hist.png", dpi=160)
plt.show()


In [ ]:
ess, ess_ratio = effective_sample_size(weights_raw)
print(f"ESS:       {ess:.0f} / {len(weights_raw)} = {ess_ratio:.4f}")
print(f"Próg 0.05: {'OK' if ess_ratio >= 0.05 else 'KRYTYCZNY — duże ryzyko wysokiej wariancji IPS'}")


In [ ]:
diag = overlap_diagnostics(pscores_train, action, PI_EVAL, N_ACTIONS)
min_per_action = diag["min_pscore_per_action"]

fig, ax = plt.subplots(figsize=(14, 2))
bar_colors = ["red" if v < 0.001 else "steelblue" for v in np.nan_to_num(min_per_action)]
ax.bar(np.arange(N_ACTIONS), np.nan_to_num(min_per_action), color=bar_colors, width=0.8)
ax.axhline(0.001, color="red", linestyle="--", linewidth=0.8, label="threshold 0.001")
ax.set_xlabel("Action ID")
ax.set_ylabel("Min P(a|s)")
ax.set_title("Minimum propensity score per action (red = below 0.001 → overlap issue)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "overlap_per_action.png", dpi=160)
plt.show()

n_problem = int((np.nan_to_num(min_per_action) < 0.001).sum())
print(f"Akcje z min pscore < 0.001: {n_problem} / {N_ACTIONS}")


## Zastosowanie modelu na danych BTS


In [ ]:
dataset_bts = OpenBanditDataset(behavior_policy="bts", campaign="all")
feedback_bts = dataset_bts.obtain_batch_bandit_feedback()

context_bts = feedback_bts["context"][:, :N_FEATURES]
action_bts = feedback_bts["action"]

pscores_bts = extract_propensity_scores(
    pscore_model, feedback_bts["context"], action_bts, n_features=N_FEATURES
)

print(f"BTS pscores — shape: {pscores_bts.shape}")
print(f"  min={pscores_bts.min():.6f}  max={pscores_bts.max():.6f}  mean={pscores_bts.mean():.4f}")
print(f"  % NaN:         {np.isnan(pscores_bts).mean()*100:.2f}%")
print(f"  % below 0.001: {(pscores_bts < 0.001).mean()*100:.2f}%")

weights_bts = PI_EVAL / np.clip(pscores_bts, 1e-9, None)
ess_bts, ess_ratio_bts = effective_sample_size(weights_bts)
print(f"ESS (BTS): {ess_bts:.0f} / {len(pscores_bts)} = {ess_ratio_bts:.4f}")

RESULTS_DIR = Path("../results/week5")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
np.save(RESULTS_DIR / "pscores_bts.npy", pscores_bts)
print(f"Saved -> {RESULTS_DIR / 'pscores_bts.npy'}")


In [ ]:
from IPython.display import display

summary = pd.DataFrame({
    "Metryka": [
        "Multiclass logloss",
        "Top-1 accuracy",
        "ESS ratio (train)",
        "ESS ratio (BTS)",
        "Akcje z overlap problem (<0.001)",
    ],
    "Wartość": [
        f"{logloss:.4f}",
        f"{acc:.4f}",
        f"{ess_ratio:.4f}",
        f"{ess_ratio_bts:.4f}",
        f"{n_problem} / {N_ACTIONS}",
    ],
})
display(summary)

for label, ok in [
    ("src/propensity.py — train + extract", True),
    ("ESS ratio (train) > 0.05", ess_ratio >= 0.05),
    ("ESS ratio (BTS) > 0.05", ess_ratio_bts >= 0.05),
    ("Brak NaN w pscores_bts", np.isnan(pscores_bts).sum() == 0),
    ("pscores_bts zapisane", (RESULTS_DIR / "pscores_bts.npy").exists()),
]:
    print(f"[{'x' if ok else ' '}] {label}")
